In [ ]:
import yfinance as yf
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Input
def client_input():
    '''
    purpose: function that gathers input when needed
    parameters: None
    return: tickers (str) - used to scrape information. 
            period (str) - used to gather time period of information.
    '''

    # Initialize empty list to hold ticker symbols
    l_tick = []
    
    # While loop to gather ticker symbols from user input
    while True:
        ticker = input("Enter ticker symbol: ").upper()

        # Breaks loop if user entry is empty
        if ticker == "":
            break
        
        # Adds ticker to the end of the 'tickers' list
        l_tick.append(ticker)

    # Gathers period of time for which to gather information
    period = input("Enter period (e.g. 2y, 1y, 6mo, 3mo, 1mo): ")

    return l_tick, period

In [ ]:
# Pull data
def pull_data(l_ticker, period):
    '''
    purpose: get raw historical price data from one ticker from yfinance
    parameter: ticker_symbol (str) - e.g. "AAPL". period (str) - e.g. "2y".
    return: hist (DataFrame) - daily price data, indexed by date. 
    '''

    # Create an empty dictionary to store the called data for each ticker
    d_tick = {}
    
    # Loop through each ticker in the list of tickers and pull historical data for each
    for ticker in l_ticker:

        # Scrape information of current element (ticker) in the list of tickers
        ticker_scrape = yf.Ticker(ticker)
        hist = ticker_scrape.history(period = period)

        # Clean Data - but this is inefficient and can rid of necessary information
        hist = hist.dropna(subset = ['Close'])

        # Add the historical DataFrame to the market_data dictionary with the ticker symbol as the key
        d_tick[ticker] = hist

    return d_tick

In [ ]:
# I want each inputed ticker, to scrape the data for its relevant object. 
# Then it will carry its own attributes, this mainly being the dataframe.
# But also will have defining attributes. 
class Security:
    '''
    Attributes: name - type: str - name of corresponding security.
                market_data - type: array - dataframe of corresponding security.
    '''
    def __init__(self, name, raw_market_data):
        self.name = name
        self.raw_market_data = raw_market_data

        # Intialize DataFrame
        self.indicators = pd.DataFrame(index = raw_market_data.index)
        self.signals = pd.DataFrame(index = raw_market_data.index)

    def getMarketData(self):
        return self.raw_market_data

    def __str__(self):
        return self.name

# tuple unpacking
l_ticker, period = client_input()
d_tick = pull_data(l_ticker, period)

# declare securities list - will store objects
securities = []

# for loop through list of tickers to instance objects, providing name, and the corresponding dataframe
for ticker in l_ticker:
    ticker = Security(ticker, d_tick[ticker])
    securities.append(ticker)

In [ ]:
# Indicator Packages

# Simple Moving Averages (SMA)
def s_moving_avg(ticker, df, period_1, period_2):
    ticker.indicators[f'MA{period_1}'] = df['Close'].rolling(period_1).mean()
    ticker.indicators[f'MA{period_2}'] = df['Close'].rolling(period_2).mean()

# Calculate Relative-Strength Index (RSI)
def rsi(ticker, df, avg):
    delta = df['Close'].diff()

    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    avg_gain = gain.rolling(avg).mean()
    avg_loss = loss.rolling(avg).mean()

    rs = avg_gain / avg_loss

    ticker.indicators['RSI'] = 100 - (100 / (1+rs))

In [ ]:
# Signal Packages

def ma_signal(ticker, period_1, period_2):
    fast = ticker.indicators[f'MA{period_1}']
    slow = ticker.indicators[f'MA{period_2}']

    ticker.signals['golden_cross'] = (fast > slow) & (fast.shift(1) <= slow.shift(1))
    ticker.signals['death_cross'] = (fast < slow) & (fast.shift(1) >= slow.shift(1))

# RSI Signal
def rsi_signal(ticker, oversold, overbought):
    rsi_values = ticker.indicators['RSI']

    ticker.signals['oversold'] = rsi_values <= oversold
    ticker.signals['overbought'] = rsi_values >= overbought

In [ ]:
# Plugin Registries

INDICATOR_PLUGINS = {
    "moving_average": s_moving_avg,
    "rsi": rsi
}

SIGNAL_PLUGINS = {
    "moving_average_cross": ma_signal,
    "rsi_threshold": rsi_signal
}

In [ ]:
# An Order is your instruction to the market
class Order:
    def __init__(
        self,
        security,       # security we wish to purchase
        order_side,     # whether we wish to buy/sell
        order_type,     # whether we choose to place a MARKET or LIMIT order
        quantity_type,  # whether we want our order to placed in SHARE or VALUE
        size,           # from quantity_type then we will ask for the size of SHARES or VALUE
        timestamp       # time of our order placed
    ):
        self.security = security
        self.order_side = order_side
        self.order_type = order_type
        self.quantity_type = quantity_type
        self.size = size
        self.timestamp = timestamp
        self.status = 'PENDING'

In [ ]:
# Execution is the market's response to that instruction
class Execution:
    '''
    Attributes: order - type: Order - order that this execution fufills.
                size - type: int/float - quantity of the order that is executed.
                price - type: float - price at which the order is executed.
                timestamp - type: dataframe - time at which the order was executed.
    '''
    def __init__(
        self, 
        order,
        size,
        price,
        timestamp
    ):
        self.order = order
        self.size = size
        self.price = price
        self.timestamp = timestamp

In [ ]:
# Transaction records the completed buy or sell in a form that allows us to reconstruct exactly 
# what happened to the portfolio.
class Transaction:
    '''
    Attributes: account - type: Portfolio - account that the transaction occured in.
                security - type: Security - security that was bought or sold. 
                order_side - type: str - whether the transaction was a BUY or SELL. 
                order_type - type: str - type of order that was executed.
                size - type: int/float - number of units bought or sold.
                price - price: float - execution price per unit in the security's denominated currency.
                currency - type: str - currency in which the security was denominated. 
                fx_rate - type: float - exchange rate used to convert the transaction into account's currency.
                total_cost - type: float - total amount applied to the account from the transaction.
                timestamp - type: datetime - time at which the transaction occured. 
    '''
    def __init__(
        self,
        account,
        security,
        order_side,
        order_type,
        size,
        price,
        currency,
        fx_rate,
        total_cost,
        timestamp
    ):
        self.account = account
        self.security = security
        self.order_side = order_side
        self.order_type = order_type
        self.order_side = order_side
        self.size = size
        self.price = price
        self.currency = currency
        self.fx_rate = fx_rate
        self.total_cost = total_cost
        self.timestamp = timestamp

In [ ]:
class Portfolio:
    def __init__(self, funds):
        self.funds = funds

        # Positions (dict) - will provide the security and of positions it has taken.
        # (e.g. AAPL -> 10 shares)
        self.positions = {}

        # Order placed
        self.orders = []

        # Completed transactions
        self.transactions = []

        # Metrics (df) - will compile given metrics of the portfolio. These metrics should also be plugins. 
        self.metrics = {}

    def buy(self, security, size, timestamp):
        order = Order(
            security = security,
            order_side = 'BUY',
            order_type = 'MARKET',
            quantity_type = 'SHARES',
            size = size,
            timestamp = timestamp
        )

        self.orders.append(order)
        return order

    def sell(self, security, size, timestamp):
        order = Order(
            security = security,
            order_side = 'SELL',
            order_type = 'MARKET',
            quantity_type = 'SHARES',
            size = size,
            timestamp = timestamp            
        )

        self.orders.append(order)

        return order

    def process_order(self, order):
        if order.order_type == 'MARKET':
            execution = market(order, order.timestamp)

        elif order.order_type == 'LIMIT':
            raise NotImplementedError("LIMIT orders are not implemented yet.")

        else:
            raise ValueError(f"Unsupported order type: {order.order_type}")

        transaction = Transaction(
            account = self,
            security = execution.order.security,
            order_side = execution.order.order_side,
            order_type = execution.order.order_type,
            size = execution.size,
            price = execution.price,

            currency = None
            fx_rate = None
            total_cost = None,

            timestamp = execution.timestop
        )

        self.transactions.append(transaction)

        self.update_position(transaction)
        self.update_funds(transaction)
        
        return transaction

    def update_position(self, transaction):
        security = transaction.security

        if security not in self.positions:
            self.positions[security] = 0

        if transaction.order_side == 'BUY':
            self.positions[security] += transaction.size

        elif transaction.order_side == 'SELL':
            self.positions[security] -= transaction.size

        else:
            raise ValueError(f"Unsupported order side: {transaction.order_side}")

    def update_funds(self, transaction):
        value = transaction.size * transaction.price

        if transaction.order_side == 'BUY':
            self.funds -= value

        elif transaction.order_side == 'SELL':
            self.funds += value

In [ ]:
# Order Types 
def market(order, timestamp):
    market_data = order.security.getMarketData()

    price = market_data.loc[timestamp, 'Close']

    execution = Execution(
        order = order,          # preserves the link back to the instruction that produced execution
        size = order.size,      # current platform assumes the requested order is fully executed
        price = price,          # simulated execution price from available market data
        timestamp = timestamp   # execution time, rather order's timestamp
        )

    order.status = 'FILLED'

    return execution

def limit(order, market_data, limit_price):
    pass

In [ ]:
class Strategy:
    def __init__(self, indicators, signals):
        self.indicators = indicators
        self.signals = signals

In [ ]:
# Test Strategy
strategy = Strategy(indicators = [
    {"plugin": "moving_average",
     "parameters": {"period_1": 50, "period_2": 200}},

     {"plugin": "rsi",
      "parameters": {"avg": 14}} 
    ],

signals = [
    {"plugin": "moving_average_cross",
    "parameters": {"period_1": 50, "period_2": 200}},

    {"plugin": "rsi_threshold",
     "parameters": {"oversold": 30, "overbought": 70}}
    ]
)

# Apply strategy to securities
for security in securities:
    strategy.apply(security)

In [ ]:
def simulate(security, strategy, portfolio):
    '''
    purpose: simulate the strategy through each historical period without allowing future market
             data to influence the current period (t).
    parameters: 
        security - security object being simulated.
        strategy - strategy object containing indicator and signal plugins.
        portfolio - portfolio object placing and receiving orders.
    '''
    market_data = security.getMarketData()

    for timestamp in security.getMarketData().index:
        # Only provide the strategy information available at the time.
        current_data = security.getMarketData().loc[:timestamp]

    # Calculate each indicator using the current information set.
    for indicator in strategy.indicators:
        plugin = INDICATOR_PLUGINS[indicator["plugin"]]
        plugin(security, current_data, **indicator["parameters"])

    # Generate the strategy's signals from the indicators calculated previously.
    for signal in strategy.signals:
        plugin = SIGNAL_PLUGINS[signal["plugin"]]
        plugin(security, **signal["parameters"])

    # Pick up the signals generated for this timestamp.
    signals = security.signals.loc[timestamp]

    # Any True signal causes the Portfolio to create an order
    for signal, active in signals.items():
            if active:
                 # Order action for each signal is handled by strategy
                 pass
            
    return portfolio
    

In [ ]:
# Engine

# Outer function
def run_backtest(market_data, initial_cash = 1000):
    '''
    purpose: walk through data day by day, simulating buys and sells.
    parameters: hist(DataFrame) - must contain buy_signal and sell_signal columns. 
                initial_cash(float) - starting cash.
    return: cash(float), shares(float), book_value(float) - ending state after the full simulation.
    '''
    # Inner function - runs buy/sell signal of given ticker
    def simulate(df, initial_cash):

        cash = initial_cash
        shares = 0
        book_value = 0
        shares_bought = 0

        for date, row in df.iterrows():
            price = row['Close']

            if row['buy_signal'] and cash >= price:
                spend = price
                shares_bought = spend / price
                shares += shares_bought
                cash -= spend
                book_value += spend
                print("\n", date, "BUY", round(shares, 2), "shares @", "$",round(price, 2))

            elif row['sell_signal'] and shares > 0:
                proceeds = shares * price
                realized_value = proceeds - book_value
                cash += proceeds
                shares = 0
                book_value = 0
                print("\n", date, "SELL", round(shares_bought, 2), "shares @", "$", round(price, 2), "Realized gain/loss:", round(realized_value, 2))

        return cash, shares, book_value

    # Create a dictonary that contains information on each ticker's holdings
    holdings = {}

    for ticker, df in market_data.items():
        cash, shares, book_value = simulate(df, initial_cash)
        holdings[ticker] = (cash, shares, book_value)

    return holdings

In [ ]:
def plot_ticker(df, ticker):
    '''
    purpose: plot the closing price, MAs, and buy/sell signals for a given ticker.
    parameters: hist (DataFrame) - must contain Close, MA50, MA200, buy_signal, and sell_signal columns. 
                ticker_symbol (str) - used for chart title.
    return: None
    '''
    plt.figure(figsize = (12, 6))
    plt.plot(df.index, df['Close'], label = 'Close', alpha = 0.7)
    plt.plot(df.index, df['MA50'], label = 'MA50', color = "green")
    plt.plot(df.index, df['MA200'], label = 'MA200', color = "red")

    buys = df[df['buy_signal']]
    sells = df[df['sell_signal']]

    plt.scatter(buys.index, buys['Close'], 
                color = 'green', marker = '^', s = 100, label = 'Buy', zorder = 5)
    plt.scatter(sells.index, sells['Close'], 
                color = 'red', marker = 'v', s = 100, label = 'Sell', zorder = 5)
    
    plt.title(ticker)
    plt.legend()
    plt.show()

In [ ]:
# Report
def summarize_results(market_data, holdings, initial_cash):
    '''
    purpose: turn ending cash/shares/book_value into readable results.
    parameters: hist(DataFrame), cash(float), shares(float), book_value(float), initial_cash(float)
    return: None - this function only prints
    '''
    for ticker, df in market_data.items():
        cash, shares, book_value = holdings[ticker]

        final_price = df['Close'].iloc[-1]
        current_value_of_holdings = shares * final_price
        portfolio_value = cash + current_value_of_holdings
        overall_return = (portfolio_value - initial_cash) / initial_cash

        print(f"\n--- Summary: {ticker} ---")
        print(f"Available cash: ${cash:,.2f}")
        print(f"Current value of holdings: ${current_value_of_holdings:,.2f}")

        if shares > 0:
            holding_return = (current_value_of_holdings - book_value) / book_value
            holding_gain_loss = current_value_of_holdings - book_value

            print(f"Book Value (cost basis) of holdings: ${book_value:,.2f}")
            print(f"Return on holdings: {holding_return:.2%}")
            print(f"Gain/loss on holdings: ${holding_gain_loss:,.2f}")

        print(f"Total portfolio value: ${portfolio_value:,.2f}")
        print(f"Overall return (start to end): {overall_return:.2%}")


In [ ]:
# Main function
def run_engine():
    '''
    purpose: call the helper functions to run the backtest engine.
    parameters: None
    return: None - this function only prints
    '''
    tickers, period = client_input()
    market_data = pull_data(ticker_series, period)
    market_data = calculate_indicators(market_data)
    market_data = generate_signals(market_data)

    for ticker, df in market_data.items():
        plot_ticker(df, ticker)

    holdings = run_backtest(market_data, initial_cash = 1000)
    summarize_results(market_data, holdings, initial_cash = 1000)

# actually runs the main function
run_engine()